# Pseudo-Supervised CIFAR-10 Baseline

Single-episode pseudo-supervised training on the shared N-pool, followed by spectral and within-pool KNN diagnostics.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is only available inside Google Colab; skipping Drive mount.')


In [ ]:
import getpass
import os
import subprocess
from pathlib import Path
from urllib.parse import quote

PUBLIC_REPO_URL = 'https://github.com/moucheng2017/instance_self_sup.git'
BRANCH = '2-vicreg-barlow-twins-baselines'  # Use 'main' after this baseline suite is merged.
REPO_DIR = Path('/content/instance_self_sup')
GITHUB_TOKEN = ''  # Set to a token string here if the repo is private.


def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get('GITHUB_TOKEN')
except Exception:
    pass

repo_url = PUBLIC_REPO_URL
if GITHUB_TOKEN:
    repo_url = PUBLIC_REPO_URL.replace('https://', f'https://oauth2:{quote(GITHUB_TOKEN, safe="")}@')


def sync_repo():
    if GITHUB_TOKEN:
        run(['git', 'remote', 'set-url', 'origin', repo_url], cwd=REPO_DIR)
    else:
        print('No GitHub token found; keeping the existing origin URL for fetch/pull.')
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)

os.chdir('/content')

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, repo_url, str(REPO_DIR)])
else:
    sync_repo()

os.chdir(REPO_DIR)
run(['python', '-m', 'pip', 'install', '-r', 'requirements_colab.txt'])
print('Repository is ready at', REPO_DIR)


## Sweep Parameters

| Parameter | Description |
|---|---|
| `N_SWEEP`, `subset_n`, `subset_seed` | Shared low-data pool for training and diagnostics |
| `BATCH_SIZE_SWEEP`, `NUM_EPOCHS`, `STOP_AT_EPOCH` | Per-N training controls; batch size is clamped to `N` |
| `SAVING_FREQUENCY` | Save a checkpoint every N epochs; checkpoint filename includes the epoch number |
| `INIT_CHECKPOINT`, `INIT_LOAD_BACKBONE`, `INIT_LOAD_PROJECTOR`, `INIT_LOAD_PREDICTOR` | Optional checkpoint initialization controls |
| `knn_monitor` / `monitor_accuracy` | In-training monitor is disabled by default; Section 2.1 KNN is computed in the diagnostics cell |
| `cosine_softmax`, `l2_norm_backbone_features`, `negatives_ratio` | Native single-episode pseudo-supervised settings |


In [ ]:
from colab_utils import train_from_colab

import os

# --- Weights & Biases setup (mirrors the GITHUB_TOKEN pattern above) ---
# Easiest: add your key to Colab Secrets (left sidebar key icon) named WANDB_API_KEY,
# or paste it below. Leave blank to run without W&B.
WANDB_API_KEY = ''
WANDB_ENTITY = None  # your W&B team/username, or None for your default account
try:
    from google.colab import userdata
    WANDB_API_KEY = WANDB_API_KEY or userdata.get('WANDB_API_KEY')
except Exception:
    pass
USE_WANDB = bool(WANDB_API_KEY)
if USE_WANDB:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
    try:
        import wandb
        wandb.login()
        print('W&B login OK.')
    except Exception as exc:
        USE_WANDB = False
        print('W&B login failed; continuing without W&B:', exc)
else:
    print('No WANDB_API_KEY found; W&B logging disabled for this run.')

CONFIG_FILE = 'configs/baselines/pseudo_sup_cifar10.yaml'  # config_file='configs/baselines/pseudo_sup_cifar10.yaml'
PROJECT_NAME = 'InstanceSelfSup_CIFAR10'
DATA_DIR = '/content/drive/MyDrive/cifar10_data'  # shared CIFAR-10 location, reused across projects (downloaded once)
SUBSET_SEED = 42
N_SWEEP = [1000, 5000] # 'full' means using the full dataset (50k training examples)
BATCH_SIZE_SWEEP = [512]  # swept like N_SWEEP; each batch is clamped to <= N per pool size
NUM_EPOCHS = 200
STOP_AT_EPOCH = 200
SAVING_FREQUENCY = 100  # save a checkpoint every N epochs; filename includes the epoch number
BASE_LR = 0.05  # shared base LR across all methods (all use SGD); lowered from the original 0.3 that diverged
GRAD_CLIP = 1.0  # gradient-norm clipping; set to None to disable
WEIGHT_DECAY = 5e-4  # SGD weight decay, shared across all methods
WARMUP_EPOCHS = 10  # LR warmup epochs, shared across all methods
BASE_SAMPLES_PER_EPOCH = 80000  # base per-epoch sample budget at batch size 256; scales with batch as BASE_SAMPLES_PER_EPOCH * batch_size // 256 (so iters/epoch stays constant across batch sizes)

INIT_CHECKPOINT = None
INIT_LOAD_BACKBONE = True
INIT_LOAD_PROJECTOR = False
INIT_LOAD_PREDICTOR = False

COSINE_SOFTMAX = True
L2_NORM_BACKBONE_FEATURES = False
NEGATIVES_RATIO = 0.25


def batch_size_for_n(n, default_batch_size):
    if n == 'full' or n is None:
        return default_batch_size
    return min(default_batch_size, int(n))


def samples_per_epoch_for_n(n, batch_size):
    return max(BASE_SAMPLES_PER_EPOCH * batch_size // 256, batch_size)


SWEEP = []
_seen_combos = set()
for _n in N_SWEEP:
    for _default_bs in BATCH_SIZE_SWEEP:
        _bs = batch_size_for_n(_n, _default_bs)
        if (_n, _bs) not in _seen_combos:
            _seen_combos.add((_n, _bs))
            SWEEP.append((_n, _bs))

import glob
import re

RESUME = True  # auto-resume each (N, batch) run from its latest saved checkpoint if one exists (e.g. after a Colab interruption)
CKPT_DIR = f'/content/drive/MyDrive/{PROJECT_NAME}/checkpoints'


def latest_checkpoint_for(run_name):
    paths = glob.glob(os.path.join(CKPT_DIR, f'{run_name}_epoch*.pth'))
    def _epoch(path):
        m = re.search(r'_epoch(\d+)_', os.path.basename(path))
        return int(m.group(1)) if m else -1
    return max(paths, key=_epoch) if paths else None


results = []
for n, batch_size in SWEEP:
    subset_n = None if n == 'full' else int(n)
    samples_per_epoch = samples_per_epoch_for_n(n, batch_size)
    run_name = f'{CONFIG_FILE.split("/")[-1].replace(".yaml", "")}-N{n}-bs{batch_size}'
    resume_checkpoint = latest_checkpoint_for(run_name) if RESUME else None
    if resume_checkpoint:
        print(f'Resuming {run_name} from {resume_checkpoint}')
    overrides = {
        'name': run_name,
        'logger': {
            'wandb': USE_WANDB,
            'wandb_project': PROJECT_NAME,
            'wandb_entity': WANDB_ENTITY,
        },
        'train': {
            'subset_n': subset_n,
            'resume_checkpoint': resume_checkpoint,
            'subset_seed': SUBSET_SEED,
            'batch_size': batch_size,
            'num_epochs': NUM_EPOCHS,
            'stop_at_epoch': STOP_AT_EPOCH,
            'base_lr': BASE_LR,
            'grad_clip': GRAD_CLIP,
            'warmup_epochs': WARMUP_EPOCHS,
            'optimizer': {'weight_decay': WEIGHT_DECAY},
            'saving_frequency': SAVING_FREQUENCY,
            'knn_monitor': True,
            'negatives_ratio': NEGATIVES_RATIO,
            'augment_probability': 1.0,
            'samples_per_epoch': samples_per_epoch,
        },
        'model': {
            'init_checkpoint': INIT_CHECKPOINT,
            'init_load_backbone': INIT_LOAD_BACKBONE,
            'init_load_projector': INIT_LOAD_PROJECTOR,
            'init_load_predictor': INIT_LOAD_PREDICTOR,
            'cosine_softmax': COSINE_SOFTMAX,
            'l2_norm_backbone_features': L2_NORM_BACKBONE_FEATURES,
        },
    }
    run_result = train_from_colab(
        config_file=CONFIG_FILE,
        project_name=PROJECT_NAME,
        data_dir=DATA_DIR,
        overrides=overrides,
        device='cuda',
        download=True,
        hide_progress=False,
    )
    results.append({
        'N': n,
        'subset_n': subset_n,
        'subset_seed': SUBSET_SEED,
        'batch_size': batch_size,
        'model_path': run_result['model_path'],
        'log_dir': run_result['log_dir'],
        'selected_subset_indices_path': run_result.get('selected_subset_indices_path'),
        'monitor_accuracy': run_result.get('accuracy'),
    })

results
